In [3]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys

print(sys.executable)
print(sys.version)

C:\Users\acer\anaconda3\python.exe
3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

PyTorch: 2.13.0+cpu
CUDA: False


In [4]:
import os

print(os.getcwd())
print(os.listdir())

C:\Users\acer\Downloads\semester 5\NLP SKILL\CLASS
['.ipynb_checkpoints', '5_gram_project.ipynb', 'anaconda_projects', 'Assignment_A.txt', 'Assignment_B.txt', 'Assignment_C.txt', 'Assignment_D.txt', 'Assignment_E.txt', 'Assignment_F.txt', 'docs .txt', 'hmm_pos_test_dataset.csv', 'hmm_pos_train_dataset.csv', 'IMDB Dataset.csv', 'IMDB_Dataset_CLEANED.csv', 'IMDB_Dataset_Preprocessed.csv', 'MovieReview.ipynb', 'NER dataset.csv', 'NLP_LAB_PROG.ipynb', 'NLP_Skill_5.ipynb', 'NLP_SKILL_HMMPOSTagger.ipynb', 'NLP_SKILL_Palagiarism.ipynb', 'NLP_SKILL_Spell_corr.ipynb', 'NLP_SKILL_TASK_Next_Word.ipynb', 'nltk_task.ipynb', 'Nltk_task2.ipynb', 'Nltk_task_1.ipynb', 'nltk_task_24.ipynb', 'plagiarism_similarity_report.csv', 'pos_tags.csv', 'Skill_Task_2_Pos.ipynb', 'Spelling_Error_Dataset.csv', 'Untitled.ipynb', 'Untitled1.ipynb', 'Untitled10.ipynb', 'Untitled11.ipynb', 'Untitled12.ipynb', 'Untitled13.ipynb', 'Untitled14.ipynb', 'Untitled15.ipynb', 'Untitled16.ipynb', 'Untitled17.ipynb', 'Untitled18.i

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter

print("Starting...", flush=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device, flush=True)

df = pd.read_csv("DataSet_.csv")

print("Dataset loaded:", df.shape, flush=True)

df = df.dropna()

df["review"] = df["review"].astype(str)

df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()

df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

df = df.dropna(subset=["label"])

df["label"] = df["label"].astype(int)

print("Positive:", (df["label"] == 1).sum(), flush=True)
print("Negative:", (df["label"] == 0).sum(), flush=True)


def clean_text(text):
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


print("Cleaning reviews...", flush=True)

df["clean_review"] = df["review"].apply(clean_text)

print("Cleaning completed.", flush=True)


train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Training:", len(train_df), flush=True)
print("Validation:", len(validation_df), flush=True)
print("Testing:", len(test_df), flush=True)


print("Creating vocabulary...", flush=True)

counter = Counter()

for text in train_df["clean_review"]:
    counter.update(text.split())

max_vocab_size = 8000

word_to_index = {
    "<PAD>": 0,
    "<UNK>": 1
}

for i, (word, count) in enumerate(
    counter.most_common(max_vocab_size - 2),
    start=2
):
    word_to_index[word] = i

vocab_size = len(word_to_index)

print("Vocabulary size:", vocab_size, flush=True)


max_length = 60


def encode_text(text):

    words = text.split()

    sequence = [
        word_to_index.get(word, 1)
        for word in words[:max_length]
    ]

    sequence += [0] * (
        max_length - len(sequence)
    )

    return sequence


print("Encoding reviews...", flush=True)

train_sequences = [
    encode_text(text)
    for text in train_df["clean_review"]
]

validation_sequences = [
    encode_text(text)
    for text in validation_df["clean_review"]
]

test_sequences = [
    encode_text(text)
    for text in test_df["clean_review"]
]

print("Encoding completed.", flush=True)


class ReviewDataset(Dataset):

    def __init__(self, sequences, labels):

        self.x = torch.tensor(
            sequences,
            dtype=torch.long
        )

        self.y = torch.tensor(
            labels,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]


train_dataset = ReviewDataset(
    train_sequences,
    train_df["label"].tolist()
)

validation_dataset = ReviewDataset(
    validation_sequences,
    validation_df["label"].tolist()
)

test_dataset = ReviewDataset(
    test_sequences,
    test_df["label"].tolist()
)


batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)


class LSTMModel(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            32,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            32,
            32,
            batch_first=True
        )

        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self, x):

        x = self.embedding(x)

        output, (hidden, cell) = self.lstm(x)

        hidden = hidden[-1]

        output = self.fc(hidden)

        return output.squeeze(1)


print("Building LSTM model...", flush=True)

model = LSTMModel(
    vocab_size
).to(device)

print(model, flush=True)


criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 2

train_losses = []

validation_losses = []

best_loss = float("inf")


print("\nTRAINING STARTED", flush=True)
print("==============================", flush=True)


for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch_number, (reviews, labels) in enumerate(
        train_loader
    ):

        reviews = reviews.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(reviews)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        if batch_number % 20 == 0:

            print(
                f"Epoch {epoch + 1} | "
                f"Batch {batch_number + 1}/{len(train_loader)}",
                flush=True
            )

    train_loss = (
        total_loss /
        len(train_loader)
    )

    model.eval()

    total_validation_loss = 0

    with torch.no_grad():

        for reviews, labels in validation_loader:

            reviews = reviews.to(device)

            labels = labels.to(device)

            outputs = model(reviews)

            loss = criterion(
                outputs,
                labels
            )

            total_validation_loss += loss.item()

    validation_loss = (
        total_validation_loss /
        len(validation_loader)
    )

    train_losses.append(train_loss)

    validation_losses.append(validation_loss)

    print(
        f"\nEpoch {epoch + 1}/{epochs} "
        f"| Training Loss: {train_loss:.4f} "
        f"| Validation Loss: {validation_loss:.4f}\n",
        flush=True
    )

    if validation_loss < best_loss:

        best_loss = validation_loss

        torch.save(
            model.state_dict(),
            "lstm_sentiment_model.pth"
        )


print("Training completed.", flush=True)


model.load_state_dict(
    torch.load(
        "lstm_sentiment_model.pth",
        map_location=device
    )
)

model.eval()


print("\nEvaluating model...", flush=True)

predictions = []

actual = []

with torch.no_grad():

    for reviews, labels in test_loader:

        reviews = reviews.to(device)

        outputs = model(reviews)

        probabilities = torch.sigmoid(outputs)

        predicted = (
            probabilities >= 0.5
        ).int()

        predictions.extend(
            predicted.cpu().numpy()
        )

        actual.extend(
            labels.numpy()
        )


accuracy = accuracy_score(
    actual,
    predictions
)

precision = precision_score(
    actual,
    predictions,
    zero_division=0
)

recall = recall_score(
    actual,
    predictions,
    zero_division=0
)

f1 = f1_score(
    actual,
    predictions,
    zero_division=0
)


print("\n==============================")
print("MODEL PERFORMANCE")
print("==============================")

print(
    f"Accuracy  : {accuracy * 100:.2f}%"
)

print(
    f"Precision : {precision * 100:.2f}%"
)

print(
    f"Recall    : {recall * 100:.2f}%"
)

print(
    f"F1-Score  : {f1 * 100:.2f}%"
)


print("\n==============================")
print("TRAINING / VALIDATION LOSS")
print("==============================")

for i in range(epochs):

    print(
        f"Epoch {i + 1}: "
        f"Training Loss = {train_losses[i]:.4f}, "
        f"Validation Loss = {validation_losses[i]:.4f}"
    )


def predict_sentiment(review):

    model.eval()

    cleaned = clean_text(review)

    sequence = encode_text(cleaned)

    sequence = torch.tensor(
        [sequence],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():

        output = model(sequence)

        probability = torch.sigmoid(
            output
        ).item()

    if probability >= 0.5:

        sentiment = "Positive"

        confidence = probability * 100

    else:

        sentiment = "Negative"

        confidence = (
            1 - probability
        ) * 100

    print("\n==============================")
    print("NEW REVIEW PREDICTION")
    print("==============================")

    print(
        "Input:",
        review
    )

    print(
        "Predicted Sentiment:",
        sentiment
    )

    print(
        f"Confidence: {confidence:.2f}%"
    )


predict_sentiment(
    "The movie was excellent and very enjoyable"
)

In [1]:
print("Jupyter is working")

Jupyter is working


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cpu
Device: cpu


In [3]:
import pandas as pd

df = pd.read_csv("DataSet_.csv")

print("Dataset loaded successfully")
print(df.shape)
print(df.columns.tolist())

Dataset loaded successfully
(49396, 2)
['review', 'sentiment']


In [1]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter

print("Starting...", flush=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device, flush=True)

df = pd.read_csv("DataSet_.csv")

print("Dataset Shape:", df.shape, flush=True)

df = df.dropna()

df["review"] = df["review"].astype(str)

df["sentiment"] = (
    df["sentiment"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

df = df.dropna(subset=["label"])

df["label"] = df["label"].astype(int)

print(
    "Positive Reviews:",
    int((df["label"] == 1).sum()),
    flush=True
)

print(
    "Negative Reviews:",
    int((df["label"] == 0).sum()),
    flush=True
)


def clean_text(text):

    text = text.lower()

    text = re.sub(
        r"<br\s*/?>",
        " ",
        text
    )

    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


print("Cleaning reviews...", flush=True)

df["clean_review"] = df["review"].apply(
    clean_text
)

print("Cleaning completed.", flush=True)


train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print(
    "Training Samples:",
    len(train_df),
    flush=True
)

print(
    "Validation Samples:",
    len(validation_df),
    flush=True
)

print(
    "Testing Samples:",
    len(test_df),
    flush=True
)


print("Creating vocabulary...", flush=True)

counter = Counter()

for text in train_df["clean_review"]:

    counter.update(
        text.split()
    )


max_vocab_size = 8000

word_to_index = {
    "<PAD>": 0,
    "<UNK>": 1
}

for index, (word, count) in enumerate(
    counter.most_common(max_vocab_size - 2),
    start=2
):

    word_to_index[word] = index


vocab_size = len(word_to_index)

print(
    "Vocabulary Size:",
    vocab_size,
    flush=True
)


max_length = 60


def encode_text(text):

    words = text.split()

    sequence = []

    for word in words[:max_length]:

        sequence.append(
            word_to_index.get(
                word,
                1
            )
        )

    while len(sequence) < max_length:

        sequence.append(0)

    return sequence


print("Encoding training reviews...", flush=True)

train_sequences = []

for i, text in enumerate(
    train_df["clean_review"]
):

    train_sequences.append(
        encode_text(text)
    )

    if i % 5000 == 0:

        print(
            "Training reviews encoded:",
            i,
            flush=True
        )


print("Encoding validation reviews...", flush=True)

validation_sequences = [
    encode_text(text)
    for text in validation_df["clean_review"]
]


print("Encoding test reviews...", flush=True)

test_sequences = [
    encode_text(text)
    for text in test_df["clean_review"]
]

print("Encoding completed.", flush=True)


class ReviewDataset(Dataset):

    def __init__(
        self,
        sequences,
        labels
    ):

        self.x = torch.tensor(
            sequences,
            dtype=torch.long
        )

        self.y = torch.tensor(
            labels,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.y)

    def __getitem__(
        self,
        index
    ):

        return (
            self.x[index],
            self.y[index]
        )


train_dataset = ReviewDataset(
    train_sequences,
    train_df["label"].tolist()
)

validation_dataset = ReviewDataset(
    validation_sequences,
    validation_df["label"].tolist()
)

test_dataset = ReviewDataset(
    test_sequences,
    test_df["label"].tolist()
)


batch_size = 256


train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)


class LSTMSentimentModel(
    nn.Module
):

    def __init__(
        self,
        vocab_size
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            32,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=32,
            hidden_size=32,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self, x):

        embedded = self.embedding(x)

        output, (hidden, cell) = self.lstm(
            embedded
        )

        hidden = hidden[-1]

        output = self.fc(
            hidden
        )

        return output.squeeze(1)


print("Creating LSTM model...", flush=True)


model = LSTMSentimentModel(
    vocab_size
)

model = model.to(device)

print(
    "Model created successfully.",
    flush=True
)


criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


epochs = 2

training_losses = []

validation_losses = []

best_validation_loss = float(
    "inf"
)


print("\n==============================")
print("TRAINING STARTED")
print("==============================", flush=True)


for epoch in range(epochs):

    model.train()

    total_training_loss = 0

    print(
        "\nStarting Epoch",
        epoch + 1,
        flush=True
    )

    for batch_number, (
        reviews,
        labels
    ) in enumerate(train_loader):

        reviews = reviews.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            reviews
        )

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        total_training_loss += (
            loss.item()
        )

        if batch_number % 10 == 0:

            print(
                "Epoch",
                epoch + 1,
                "| Batch",
                batch_number + 1,
                "/",
                len(train_loader),
                flush=True
            )


    training_loss = (
        total_training_loss /
        len(train_loader)
    )


    model.eval()

    total_validation_loss = 0


    with torch.no_grad():

        for reviews, labels in validation_loader:

            reviews = reviews.to(device)

            labels = labels.to(device)

            outputs = model(
                reviews
            )

            loss = criterion(
                outputs,
                labels
            )

            total_validation_loss += (
                loss.item()
            )


    validation_loss = (
        total_validation_loss /
        len(validation_loader)
    )


    training_losses.append(
        training_loss
    )

    validation_losses.append(
        validation_loss
    )


    print(
        "\nEpoch",
        epoch + 1,
        "completed",
        flush=True
    )

    print(
        "Training Loss:",
        round(training_loss, 4),
        flush=True
    )

    print(
        "Validation Loss:",
        round(validation_loss, 4),
        flush=True
    )


    if validation_loss < best_validation_loss:

        best_validation_loss = (
            validation_loss
        )

        torch.save(
            model.state_dict(),
            "lstm_sentiment_model.pth"
        )


print("\nTraining completed.", flush=True)


model.load_state_dict(
    torch.load(
        "lstm_sentiment_model.pth",
        map_location=device
    )
)

model.eval()


print("\nEvaluating model...", flush=True)


predictions = []

actual = []


with torch.no_grad():

    for reviews, labels in test_loader:

        reviews = reviews.to(device)

        outputs = model(
            reviews
        )

        probabilities = torch.sigmoid(
            outputs
        )

        predicted = (
            probabilities >= 0.5
        ).int()

        predictions.extend(
            predicted.cpu().numpy()
        )

        actual.extend(
            labels.numpy()
        )


accuracy = accuracy_score(
    actual,
    predictions
)

precision = precision_score(
    actual,
    predictions,
    zero_division=0
)

recall = recall_score(
    actual,
    predictions,
    zero_division=0
)

f1 = f1_score(
    actual,
    predictions,
    zero_division=0
)


print("\n==============================")
print("MODEL PERFORMANCE")
print("==============================")

print(
    f"Accuracy  : {accuracy * 100:.2f}%"
)

print(
    f"Precision : {precision * 100:.2f}%"
)

print(
    f"Recall    : {recall * 100:.2f}%"
)

print(
    f"F1-Score  : {f1 * 100:.2f}%"
)


print("\n==============================")
print("TRAINING / VALIDATION LOSS")
print("==============================")


for i in range(epochs):

    print(
        f"Epoch {i + 1}: "
        f"Training Loss = "
        f"{training_losses[i]:.4f}, "
        f"Validation Loss = "
        f"{validation_losses[i]:.4f}"
    )


def predict_sentiment(
    review
):

    model.eval()

    cleaned_review = clean_text(
        review
    )

    sequence = encode_text(
        cleaned_review
    )

    sequence = torch.tensor(
        [sequence],
        dtype=torch.long
    )

    sequence = sequence.to(device)


    with torch.no_grad():

        output = model(
            sequence
        )

        probability = torch.sigmoid(
            output
        ).item()


    if probability >= 0.5:

        sentiment = "Positive"

        confidence = (
            probability * 100
        )

    else:

        sentiment = "Negative"

        confidence = (
            (1 - probability) * 100
        )


    print("\n==============================")
    print("NEW REVIEW PREDICTION")
    print("==============================")

    print(
        "Input:",
        review
    )

    print(
        "Predicted Sentiment:",
        sentiment
    )

    print(
        f"Confidence: {confidence:.2f}%"
    )


predict_sentiment(
    "The movie was excellent and very enjoyable"
)

Starting...
Device: cpu
Dataset Shape: (49396, 2)
Positive Reviews: 24698
Negative Reviews: 24698
Cleaning reviews...
Cleaning completed.
Training Samples: 34577
Validation Samples: 7409
Testing Samples: 7410
Creating vocabulary...
Vocabulary Size: 8000
Encoding training reviews...
Training reviews encoded: 0
Training reviews encoded: 5000
Training reviews encoded: 10000
Training reviews encoded: 15000
Training reviews encoded: 20000
Training reviews encoded: 25000
Training reviews encoded: 30000
Encoding validation reviews...
Encoding test reviews...
Encoding completed.
Creating LSTM model...
Model created successfully.

TRAINING STARTED

Starting Epoch 1
Epoch 1 | Batch 1 / 136
Epoch 1 | Batch 11 / 136
Epoch 1 | Batch 21 / 136
Epoch 1 | Batch 31 / 136
Epoch 1 | Batch 41 / 136
Epoch 1 | Batch 51 / 136
Epoch 1 | Batch 61 / 136
Epoch 1 | Batch 71 / 136
Epoch 1 | Batch 81 / 136
Epoch 1 | Batch 91 / 136
Epoch 1 | Batch 101 / 136
Epoch 1 | Batch 111 / 136
Epoch 1 | Batch 121 / 136
Epoch 1 